# Bragg fiber under strain

NumPy, static fiber. Both states use the same Rayleigh sample. Powers are normalized to unit input.

Run all cells with PyRayleigh, NumPy and Matplotlib installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyrayleigh import (BraggGrating, Fiber, FiberState, Field, FieldTerm,
    GuideOptics, Probe, Section, SpatialProfile, compute_response_diagnostics,
    lossless_reflector)
from pyrayleigh.constants import SPEED_OF_LIGHT as c

In [ ]:
nu0 = c/1550e-9
guide = GuideOptics(nu0, 1.4682, 1.474)
fiber = Fiber('bragg', .03, 293.15, nu0-1e10, nu0+1e10,
    sections=(Section(0., .03, guide, .05, .001),),
    elements=(lossless_reflector(.025, .1, .3),),
    gratings=(BraggGrating(.01, .01, 1550e-9, 100.),))
fiber = fiber.with_rayleigh(7, cell_length=.002)

In [ ]:
wavelength = np.linspace(1550e-9-40e-12, 1550e-9+40e-12, 501)
probe = Probe(c/wavelength)
strain = SpatialProfile([0., fiber.length], [10e-6, 10e-6])
state = FiberState(axial_strain=Field((FieldTerm(strain),)))
reference = compute_response_diagnostics(fiber, probe, loss_convention='background')
strained = compute_response_diagnostics(fiber, probe, state, loss_convention='background')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(wavelength*1e9, abs(reference.total.reflection)**2, label='reference')
ax[0].plot(wavelength*1e9, abs(strained.total.reflection)**2, label='10 microstrain')
ax[0].set_ylabel('Reflectance')
ax[1].plot(wavelength*1e9, abs(strained.total.reflection)**2, label='total')
ax[1].plot(wavelength*1e9, abs(strained.direct.reflection)**2, label='direct')
ax[1].plot(wavelength*1e9, abs(strained.residual.reflection)**2, label='residual')
ax[1].plot(wavelength*1e9, strained.reflection_interference, label='interference')
ax[1].set_ylabel('Power terms')
for axis in ax:
    axis.set_xlabel('Wavelength (nm)')
    axis.legend(frameon=False)
fig.tight_layout()
plt.show()